# Chapter 3 — KV Cache and Memory

This notebook is a self-contained companion to Chapter 3. It walks through the
three mechanics the chapter builds up:

1. Sizing the KV cache for MHA, GQA, MQA, and MLA
2. PagedAttention's block-by-block memory mapping, simulated step by step
3. Hash-based prefix caching, including the copy-on-write case

It uses the same `mini_inference.memory` package the chapter's tests and
examples use, so nothing here is a toy reimplementation — it's the real
`Block`, `BlockAllocator`, and `PrefixCache` classes from
`src/mini_inference/memory/`.


In [1]:
import sys
from pathlib import Path

# Make the mini_inference package importable when this notebook runs from
# ch03/ without the package having been `pip install -e .`-ed first.
repo_root = Path.cwd()
if not (repo_root / "src" / "mini_inference").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from mini_inference.memory import (
    Block,
    BlockAllocator,
    BlockTable,
    ModelProfile,
    PrefixCache,
    ServingConfig,
    WorkloadProfile,
    active_tokens,
    bytes_per_token,
    estimate_capacity,
    hash_block,
)


## 1. Sizing the KV cache: MHA vs. GQA vs. MQA vs. MLA

Section 3.4 showed that MHA, GQA, and MQA differ only in how many key/value
heads they store (`kv_heads`), while the rest of the memory formula —
`2 x layers x kv_heads x head_dim x dtype_bytes` — stays the same. Fix
`layers`, `head_dim`, and precision the way Llama 3 70B does, and vary only
`kv_heads` to see the effect.


In [2]:
config = ServingConfig(kv_dtype_bytes=2)  # FP16

variants = {
    "MHA (kv_heads = num_query_heads = 64)": ModelProfile(layers=80, kv_heads=64, head_dim=128),
    "GQA (kv_heads = 8, Llama 3 70B's real config)": ModelProfile(layers=80, kv_heads=8, head_dim=128),
    "MQA (kv_heads = 1)": ModelProfile(layers=80, kv_heads=1, head_dim=128),
}

context_length = 32_000
print(f"{'Variant':<48} {'KB/token':>10} {'GB @ 32K ctx':>14}")
print("-" * 76)
for name, model in variants.items():
    per_token = bytes_per_token(model, config)
    gb_at_context = per_token * context_length / 1024**3
    print(f"{name:<48} {per_token/1024:>10.2f} {gb_at_context:>14.2f}")


Variant                                            KB/token   GB @ 32K ctx
----------------------------------------------------------------------------
MHA (kv_heads = num_query_heads = 64)               2560.00          78.12
GQA (kv_heads = 8, Llama 3 70B's real config)        320.00           9.77
MQA (kv_heads = 1)                                    40.00           1.22


GQA gives an 8x reduction over MHA here (64 heads -> 8 groups) while keeping
most of MHA's representational power — which is why it's the standard choice
in Llama 3, Mistral, and Qwen. MQA goes further (all heads share one KV pair)
for another 8x, at a larger quality cost.

### Multi-head latent attention (MLA)

MLA (Section 3.4.3) doesn't fit the `kv_heads` formula at all: instead of
caching full per-head keys and values, it caches a single smaller latent
vector per token per layer, and reconstructs K/V from it on demand. The
per-token cost becomes `cache_values_per_token_per_layer x layers x dtype_bytes`
using whatever latent width the model's attention implementation defines —
there is no `kv_heads`/`head_dim` to plug in for a model that was never
trained with them.


In [3]:
# A DeepSeek-V2-style comparison: a GQA model and an MLA model with the same
# layer count, sized only by their cache-values-per-token-per-layer.
layers = 60
dtype_bytes = 2

gqa_kv_heads, gqa_head_dim = 16, 128
gqa_values_per_token_per_layer = 2 * gqa_kv_heads * gqa_head_dim  # keys + values

mla_latent_dim = 576  # illustrative compressed latent width

gqa_per_token = layers * gqa_values_per_token_per_layer * dtype_bytes
mla_per_token = layers * mla_latent_dim * dtype_bytes

print(f"GQA-style per-token cache: {gqa_per_token/1024:.2f} KB")
print(f"MLA-style per-token cache: {mla_per_token/1024:.2f} KB")
print(f"Reduction: {gqa_per_token / mla_per_token:.1f}x")


GQA-style per-token cache: 480.00 KB
MLA-style per-token cache: 67.50 KB
Reduction: 7.1x


**Try it yourself:** pick a real model's `config.json`, read off
`num_hidden_layers`, `num_key_value_heads` (falling back to
`num_attention_heads` if the field is absent, i.e. plain MHA), and
`hidden_size / num_attention_heads` for `head_dim`. Build a `ModelProfile`,
call `bytes_per_token`, and compare it against the same model if it used full
MHA. Multiply the difference by a long context and a large batch to see the
saving in gigabytes.


## 2. PagedAttention: block mapping, step by step

Section 3.3 introduced the block table: a request's logical token sequence
maps onto physical blocks that don't need to be contiguous in memory. Here we
grow a single request one token at a time with `block_size = 16` and watch
exactly when the allocator reaches for a new physical block.


In [4]:
allocator = BlockAllocator(num_blocks=8, block_size=16)

table = allocator.allocate_sequence(num_tokens=1)
print(f"token 1  -> block table {table.get_physical_block_ids()} (1 block claimed)")

for token_num in range(2, 35):
    new_block = allocator.append_token(table)
    if new_block is not None:
        print(f"token {token_num:<3} -> NEW physical block #{new_block.block_id} claimed, "
              f"block table now {table.get_physical_block_ids()}")

print()
print("Final block table:")
for logical_index, block in enumerate(table.blocks):
    print(f"  logical block {logical_index} -> physical block #{block.block_id} "
          f"({block.num_tokens}/{block.block_size} tokens filled)")


token 1  -> block table [0] (1 block claimed)
token 17  -> NEW physical block #1 claimed, block table now [0, 1]
token 33  -> NEW physical block #2 claimed, block table now [0, 1, 2]

Final block table:
  logical block 0 -> physical block #0 (16/16 tokens filled)
  logical block 1 -> physical block #1 (16/16 tokens filled)
  logical block 2 -> physical block #2 (2/16 tokens filled)


Tokens 1-16 fill physical block 0. Token 17 is the first token that doesn't
fit, so the allocator claims physical block 1. The same happens again at
token 33. The request's logical sequence is continuous, but nothing requires
blocks 0, 1, and 2 to sit next to each other in the underlying memory pool —
that's the indirection the block table buys you.


In [5]:
# A tiny visual of the physical pool: which blocks this request owns vs. which
# are still free, in an 8-block pool.
owned_ids = set(table.get_physical_block_ids())
row = "".join("#" if i in owned_ids else "-" for i in range(8))
print(f"Physical block pool (8 blocks): [{row}]")
print(f"  '#' = owned by this request, '-' = free  "
      f"({len(allocator.free_blocks)} blocks still free)")


Physical block pool (8 blocks): [###-----]
  '#' = owned by this request, '-' = free  (5 blocks still free)


## 3. Hash-based prefix caching, including copy-on-write

Section 3.5 reuses cached blocks across requests that share a prefix, by
hashing each block's tokens together with its parent block's hash so a hit on
block N implies every earlier block matched too. Two requests below share a
32-token prefix (2 blocks); Request B should be able to reuse both of
Request A's blocks without recomputing them.


In [6]:
block_size = 16
allocator = BlockAllocator(num_blocks=16, block_size=block_size)
prefix_cache = PrefixCache()

shared_prompt = list(range(32))  # two full blocks of shared tokens

# Request A: cold start. It computes the prompt and registers each block.
table_a = allocator.allocate_sequence(num_tokens=len(shared_prompt))
parent_hash = None
for block, chunk_start in zip(table_a.blocks, range(0, len(shared_prompt), block_size)):
    chunk = shared_prompt[chunk_start:chunk_start + block_size]
    parent_hash = prefix_cache.insert_block(chunk, block, parent_hash)

print("Request A block table:", table_a.get_physical_block_ids())
print("Request A ref_counts: ", [b.ref_count for b in table_a.blocks])


Request A block table: [0, 1]
Request A ref_counts:  [1, 1]


In [7]:
# Request B: arrives with the same 32-token prefix.
matched_blocks, remaining_tokens = prefix_cache.match_prefix(shared_prompt, block_size)

table_b = BlockTable(block_size=block_size)
table_b.blocks.extend(matched_blocks)  # match_prefix already bumped ref_count

print("Request B matched blocks:  ", table_b.get_physical_block_ids())
print("Tokens B still must prefill:", len(remaining_tokens))
print("Shared block ref_counts now:", [b.ref_count for b in table_a.blocks], "(both requests now own them)")


Request B matched blocks:   [0, 1]
Tokens B still must prefill: 0
Shared block ref_counts now: [2, 2] (both requests now own them)


Both requests point at the same two physical blocks, and neither had to
recompute the shared 32-token prefix a second time. Now suppose Request A
generates a token that's unique to it. Writing that token into a block with
`ref_count > 1` would silently corrupt Request B's copy — so the allocator
must copy-on-write: allocate a fresh, private block for the new token instead
of writing into the shared one.


In [8]:
new_block = allocator.append_token(table_a)

print("Copy-on-write triggered:", new_block is not None)
print("Request A block table now:", table_a.get_physical_block_ids())
print(f"New block #{new_block.block_id} is private: ref_count = {new_block.ref_count}")
print(f"The old shared block is untouched: ref_count = {table_a.blocks[1].ref_count} "
      f"(Request B still needs it for the first 16 tokens)")


Copy-on-write triggered: True
Request A block table now: [0, 1, 2]
New block #2 is private: ref_count = 1
The old shared block is untouched: ref_count = 2 (Request B still needs it for the first 16 tokens)


Request A's block table grew from `[shared, shared]` to
`[shared, shared, private]`. The first two blocks are still shared with
Request B — they hold tokens both requests need to read — and only the new
token's block is private to Request A. This is exactly the reference-counting
and copy-on-write behavior `tests/test_chapter3_memory.py::test_prefix_caching_hit_and_cow`
checks mechanically; here you can see it happen block by block.


## Summary

- **MHA/GQA/MQA** trade `kv_heads` for cache size; **MLA** replaces the whole
  per-head K/V cache with a smaller learned latent representation.
- **PagedAttention** grows a request one fixed-size block at a time instead of
  reserving its maximum possible length up front, and a block table maps the
  logical sequence onto whatever physical blocks are free.
- **Prefix caching** reuses those same physical blocks across requests that
  share a prefix by chaining block hashes, and **falls back to copy-on-write**
  the moment a shared block needs to diverge.

The `ch03/examples/` scripts apply all three ideas at a larger, more
realistic scale: `capacity_planner_cli.py` for end-to-end sizing,
`compare_static_vs_paged.py` for allocation waste across a batch of requests,
and `benchmark_prefix_caching.py` for prefix-caching savings on a shared
system prompt.
